# Identificacion con iNaturalist (Computer Vision)\n\nSugiere especie via la API de identificacion por imagen de iNaturalist (`/v1/computervision/score_image`) para los grupos que Pl@ntNet no cubre o que quedaron sin resolver:\n\n1. **Aves sin identificar**: fotos marcadas `ave` en `revision_animal_en_planta` con `especie_id IS NULL`.\n2. **Insectos sin identificar**: mismo mecanismo, decision `insecto`.\n3. **Limbo**: generos completos mandados a revision de experto (`fotos.estado='pendiente_revision'`, ej. Stanhopea, Oncidium, Laelia, Habenaria, Encyclia, Echeveria). iNaturalist tambien identifica plantas, asi que sirve para darles una pasada antes de que un experto humano las revise.\n4. **Mamifero / hongo / anfibio_reptil / planta (sin animal)**: agregado 18-sep, mismas categorias de `revision_animal_en_planta` que nunca tuvieron ningun pipeline de identificacion (ni Pl@ntNet, ni BioCLIP).\n\n**Requiere autenticacion** (a diferencia de la busqueda de taxones, que es abierta): `inaturalist_token` en `config_local.json`, sacado de `https://www.inaturalist.org/users/api_token` mientras se esta logueado con la cuenta de Naturalista ITESO. Ese token dura 24 horas, hay que sacar uno nuevo cada vez que se quiera correr esto de nuevo.\n\n**Nunca acepta nada solo**: guarda cada sugerencia (top-3) en `inaturalist_intentos`, no toca `fotos.especie_id` ni `especies`. La confirmacion sigue siendo 100% humana, igual que con Pl@ntNet y BioCLIP.\n\n**Aprendizajes de la corrida real (18-sep)**:\n- La API truena con error 500 si la imagen pesa varios MB (fotos de camara a resolucion completa) - hay que redimensionar a max 1024px antes de mandarla, igual que Pl@ntNet.\n- Tiene limite de tasa por minuto (no documentado publicamente): con 0.3s de pausa entre llamadas trono con 429 despues de ~660 fotos seguidas. Con 1s de pausa y reintento automatico (60s de espera) no volvio a pasar en un lote de ~2,100 fotos.\n- Con 6 fotos de ave ya identificadas de antes, acerto las 6 (una de ellas revelo que *Amazilia violiceps* y *Ramosomyia violiceps* son la misma especie con nombres de genero distintos en nuestro catalogo, sinonimia taxonomica real que se fusiono a mano)."

In [ ]:
import concurrent.futures
import json
import sqlite3
import time
from io import BytesIO
from pathlib import Path

import requests
from PIL import Image


def cargar_config(archivo=\"config_local.json\"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(f\"No se encontro {archivo}.\")
    return json.loads(ruta.read_text(encoding=\"utf-8\"))


config = cargar_config()
TOKEN = config[\"inaturalist_token\"]
RUTA_BD = Path(\"../../experimentos/bd/campuslab_volumen.db\")
con = sqlite3.connect(RUTA_BD)
print(f\"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})\")

con.execute(\"\"\"
    CREATE TABLE IF NOT EXISTS inaturalist_intentos (
        foto_id INTEGER PRIMARY KEY,
        grupo TEXT,
        nombre_sugerido TEXT,
        score REAL,
        taxon_id INTEGER,
        raw TEXT,
        fecha TEXT DEFAULT CURRENT_TIMESTAMP
    )
\"\"\")
con.commit()

_ITESO_DISPONIBLE = None


def red_iteso_disponible():
    global _ITESO_DISPONIBLE
    if _ITESO_DISPONIBLE is None:
        ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        futuro = ex.submit(lambda: Path(\"Y:/\").exists())
        try:
            _ITESO_DISPONIBLE = futuro.result(timeout=2)
        except concurrent.futures.TimeoutError:
            _ITESO_DISPONIBLE = False
        ex.shutdown(wait=False)
    return _ITESO_DISPONIBLE


def resolver_ruta_legible(ruta_actual):
    ruta_espejo = config.get(\"ruta_espejo_local_fotos\")
    if ruta_espejo:
        local = Path(ruta_espejo) / Path(ruta_actual).name
        if local.exists():
            return local
    if not red_iteso_disponible():
        return None
    ruta_directa = Path(ruta_actual)
    return ruta_directa if ruta_directa.exists() else None"

In [ ]:
candidatas_todas = con.execute(\"\"\"
    SELECT f.id, f.ruta_actual, 'ave' AS grupo
    FROM fotos f JOIN revision_animal_en_planta r ON r.foto_id = f.id
    WHERE r.decision='ave' AND f.especie_id IS NULL AND f.ruta_actual NOT LIKE '%.db'
      AND f.id NOT IN (SELECT foto_id FROM inaturalist_intentos)
      AND f.id NOT IN (SELECT foto_id FROM archivos_no_procesables)

    UNION ALL

    SELECT f.id, f.ruta_actual, 'insecto' AS grupo
    FROM fotos f JOIN revision_animal_en_planta r ON r.foto_id = f.id
    WHERE r.decision='insecto' AND f.especie_id IS NULL AND f.ruta_actual NOT LIKE '%.db'
      AND f.id NOT IN (SELECT foto_id FROM inaturalist_intentos)
      AND f.id NOT IN (SELECT foto_id FROM archivos_no_procesables)

    UNION ALL

    SELECT f.id, f.ruta_actual, 'planta_limbo' AS grupo
    FROM fotos f
    WHERE f.estado='pendiente_revision' AND f.ruta_actual NOT LIKE '%.db'
      AND f.id NOT IN (SELECT foto_id FROM inaturalist_intentos)
      AND f.id NOT IN (SELECT foto_id FROM archivos_no_procesables)

    UNION ALL

    SELECT f.id, f.ruta_actual, r.decision AS grupo
    FROM fotos f JOIN revision_animal_en_planta r ON r.foto_id = f.id
    WHERE r.decision IN ('mamifero', 'hongo', 'anfibio_reptil', 'planta')
      AND f.especie_id IS NULL AND f.ruta_actual NOT LIKE '%.db'
      AND f.id NOT IN (SELECT foto_id FROM inaturalist_intentos)
      AND f.id NOT IN (SELECT foto_id FROM archivos_no_procesables)
\"\"\").fetchall()
print(f\"Candidatas sin intentar todavia: {len(candidatas_todas):,}\")

candidatas = []
sin_acceso = 0
for foto_id, ruta, grupo in candidatas_todas:
    ruta_legible = resolver_ruta_legible(ruta)
    if ruta_legible is None:
        sin_acceso += 1
        continue
    candidatas.append((foto_id, ruta_legible, grupo))

from collections import Counter
print(f\"Disponibles ahorita: {len(candidatas):,}\")
print(f\"Sin acceso por ahora: {sin_acceso:,}\")
print(Counter(g for _, _, g in candidatas))"

In [ ]:
LADO_MAXIMO_PX = 1024  # sin esto la API responde 500 con fotos de camara a resolucion completa
URL_SCORE = \"https://api.inaturalist.org/v1/computervision/score_image\"


def preparar_imagen(ruta):
    with Image.open(ruta) as img:
        img = img.convert(\"RGB\")
        img.thumbnail((LADO_MAXIMO_PX, LADO_MAXIMO_PX), Image.LANCZOS)
        buf = BytesIO()
        img.save(buf, \"JPEG\", quality=85)
        return buf.getvalue()


def identificar(ruta, reintentos=3):
    contenido = preparar_imagen(ruta)
    for intento in range(reintentos):
        respuesta = requests.post(
            URL_SCORE,
            headers={\"Authorization\": f\"Bearer {TOKEN}\"},
            files={\"image\": (\"foto.jpg\", contenido, \"image/jpeg\")},
            timeout=30,
        )
        if respuesta.status_code == 429 and intento < reintentos - 1:
            print(f\"  429, esperando 60s antes de reintentar (intento {intento + 1})...\")
            time.sleep(60)
            continue
        respuesta.raise_for_status()
        return respuesta.json()"

In [ ]:
EJECUTAR_REAL = False  # <-- cambialo a True para correr de verdad
PAUSA_ENTRE_LLAMADAS_SEG = 1.0  # 18-sep: 0.3s tumbo la API con 429 despues de ~660 fotos

if not EJECUTAR_REAL:
    print(\"EJECUTAR_REAL esta en False. No se proceso nada todavia.\")
else:
    procesadas = errores = 0
    for i, (foto_id, ruta_legible, grupo) in enumerate(candidatas, 1):
        try:
            resultado = identificar(ruta_legible)
            candidatos = resultado.get(\"results\", [])
            if candidatos:
                mejor = candidatos[0]
                nombre = mejor[\"taxon\"][\"name\"]
                score = mejor.get(\"combined_score\", mejor.get(\"vision_score\"))
                taxon_id = mejor[\"taxon\"][\"id\"]
                raw = json.dumps(candidatos[:3], ensure_ascii=False)
            else:
                nombre, score, taxon_id, raw = None, None, None, json.dumps([])
            con.execute(
                \"INSERT OR REPLACE INTO inaturalist_intentos \"
                \"(foto_id, grupo, nombre_sugerido, score, taxon_id, raw) VALUES (?,?,?,?,?,?)\",
                (foto_id, grupo, nombre, score, taxon_id, raw),
            )
            con.commit()  # commit por foto: otros procesos pueden estar leyendo la BD
            procesadas += 1
        except Exception as e:
            errores += 1
            print(f\"error en foto {foto_id} ({grupo}): {e}\")
        time.sleep(PAUSA_ENTRE_LLAMADAS_SEG)
        if i % 50 == 0:
            print(f\"  {i}/{len(candidatas)} (ok={procesadas}, errores={errores})\")
    print(f\"\\nListo. procesadas={procesadas}, errores={errores}\")"

## Siguiente paso (no construido todavia)\n\nRevisar `inaturalist_intentos` ordenado por `score` descendente (columna `raw` trae las 3 mejores opciones por foto, no solo la primera). Falta decidir con Remi el mecanismo de revision: extender `revision_web/` con una pestana nueva, o una tabla aparte tipo `reestructura/clustering_especimen/`. Ninguna decision se aplica sola a `fotos`/`especies` todavia."